In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score



In [27]:

file_path = "brfss_clean_processed.parquet"

df = pd.read_parquet(file_path, engine="pyarrow")



In [28]:
import pandas as pd

cols = [
    "Age_group",
    "Sex",
    "Education_level",
    "Income_Cat",
    "Physical_Activity",
    "Smoker",
    "Heavy_Drinker",
    "BMI",
    "CHCKDNY2",
    "_DRDXAR2",
    "DIABTYPE"
]

df_subset = df[cols].copy()

df_subset = df_subset[df_subset["DIABTYPE"].isin([1, 2])].copy()

df_subset["DIABTYPE"] = df_subset["DIABTYPE"].map({
    1: 0,
    2: 1
})

df_subset = df_subset.replace([7, 9, 77, 99, 999, 9999], pd.NA)
df_subset = df_subset.dropna()




df_subset.head()

,Age_group,Sex,Education_level,Income_Cat,Physical_Activity,Smoker,Heavy_Drinker,BMI,CHCKDNY2,_DRDXAR2,DIABTYPE
45586,Age 65 or older,Female,Attended College or Technical School,"$25,000 < $35,000",Had physical activity,No,No,0.492596,2.0,1.0,1
45594,Age 65 or older,Female,Attended College or Technical School,"$50,000 < $100,000",No physical activity,No,No,0.524505,1.0,1.0,1
45596,Age 65 or older,Male,Graduated from College or Technical School,"$50,000 < $100,000",Had physical activity,No,No,0.356830,2.0,1.0,1
45598,Age 65 or older,Female,Graduated High School,"$50,000 < $100,000",Had physical activity,No,Yes,0.182482,2.0,1.0,1
45599,Age 65 or older,Female,Graduated High School,"$35,000 < $50,000",No physical activity,No,No,0.421898,2.0,1.0,1


In [42]:
import pandas as pd

df_encoded = df_subset.copy()



df_encoded["Sex"] = df_encoded["Sex"].map({
    "Male": 1,
    "Female": 0
})


df_encoded["Age_group"] = df_encoded["Age_group"].map({
    "Age 18 to 24": 0,
    "Age 25 to 34": 1,
    "Age 35 to 44": 2,
    "Age 45 to 54": 3,
    "Age 55 to 64": 4,
    "Age 65 or older": 5
})


df_encoded["Education_level"] = df_encoded["Education_level"].map({
    "Did not graduate High School": 0,
    "Graduated High School": 1,
    "Attended College or Technical School": 2,
    "Graduated from College or Technical School": 3
})


df_encoded["Income_Cat"] = df_encoded["Income_Cat"].map({
    "Less than $15,000": 0,
    "$15,000 < $25,000": 1,
    "$25,000 < $35,000": 2,
    "$35,000 < $50,000": 3,
    "$50,000 < $100,000": 4,
    "$100,000 or more": 5
})


df_encoded["Physical_Activity"] = df_encoded["Physical_Activity"].map({
    "Had physical activity": 1,
    "No physical activity": 0
})


df_encoded["Smoker"] = df_encoded["Smoker"].map({
    "Yes": 1,
    "No": 0
})


df_encoded["Heavy_Drinker"] = df_encoded["Heavy_Drinker"].map({
    "Yes": 1,
    "No": 0
})


df_encoded["CHCKDNY2"] = df_encoded["CHCKDNY2"].map({
    1: 0,
    2: 1
})

df_encoded["_DRDXAR2"] = df_encoded["_DRDXAR2"].map({
    1: 0,
    2: 1
})


df_encoded["BMI"] = pd.to_numeric(df_encoded["BMI"], errors="coerce")

df_encoded = df_encoded.dropna()
df_encoded



,Age_group,Sex,Education_level,Income_Cat,Physical_Activity,Smoker,Heavy_Drinker,BMI,CHCKDNY2,_DRDXAR2,DIABTYPE
45586,5,0,2,2.0,1,0,0,0.492596,1,0,1
45594,5,0,2,4.0,0,0,0,0.524505,0,0,1
45596,5,1,3,4.0,1,0,0,0.356830,1,0,1
45598,5,0,1,4.0,1,0,1,0.182482,1,0,1
45599,5,0,1,3.0,0,0,0,0.421898,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...
409360,5,1,3,4.0,1,0,0,0.294056,1,0,1
409376,5,1,0,0.0,0,0,0,0.364755,1,1,1
409391,1,1,2,2.0,1,0,0,0.284254,1,1,1
409408,4,1,3,4.0,0,0,0,0.388321,1,1,1


In [47]:
df_encoded["DIABTYPE"].value_counts()

,count
DIABTYPE,
1,7898
0,870


In [50]:

df_type1 = df_encoded.copy()
df_type1["TARGET"] = (df_type1["DIABTYPE"] == 0).astype(int)
df_type1 = df_type1.drop("DIABTYPE", axis=1)


df_type2 = df_encoded.copy()
df_type2["TARGET"] = (df_type2["DIABTYPE"] == 1).astype(int)
df_type2 = df_type2.drop("DIABTYPE", axis=1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class HealthDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class DiabetesDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()

def train_model(df, name):

    X = df.drop("TARGET", axis=1)
    y = df["TARGET"]


    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_ds = HealthDataset(X_train, y_train)
    test_ds = HealthDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

    model = DiabetesDNN(X_train.shape[1]).to(device)

    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()

    pos_weight = torch.tensor([neg / (pos + 1e-6)], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)


    epochs = 5

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


    model.eval()
    probs, labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)

            logits = model(X_batch)
            p = torch.sigmoid(logits).cpu().numpy()

            probs.extend(p)
            labels.extend(y_batch.numpy())

    probs = np.array(probs)
    labels = np.array(labels)
    preds = (probs > 0.5).astype(int)

    print(f"\n===== {name} RESULTS =====")
    print("Accuracy:", accuracy_score(labels, preds))
    print("ROC AUC :", roc_auc_score(labels, probs))
    print("F1 Score:", f1_score(labels, preds))
    print("Recall  :", recall_score(labels, preds))



train_model(df_type1, "TYPE 1 DIABETES MODEL")
train_model(df_type2, "TYPE 2 DIABETES MODEL")

[TYPE 1 DIABETES MODEL] Epoch 1/5 | Loss: 1.2516
[TYPE 1 DIABETES MODEL] Epoch 2/5 | Loss: 1.2455
[TYPE 1 DIABETES MODEL] Epoch 3/5 | Loss: 1.2389
[TYPE 1 DIABETES MODEL] Epoch 4/5 | Loss: 1.2269
[TYPE 1 DIABETES MODEL] Epoch 5/5 | Loss: 1.2189

===== TYPE 1 DIABETES MODEL RESULTS =====
Accuracy: 0.5501710376282782
ROC AUC : 0.6547231922013677
F1 Score: 0.22571148184494602
Recall  : 0.6609195402298851
[TYPE 2 DIABETES MODEL] Epoch 1/5 | Loss: 0.1376
[TYPE 2 DIABETES MODEL] Epoch 2/5 | Loss: 0.1371
[TYPE 2 DIABETES MODEL] Epoch 3/5 | Loss: 0.1366
[TYPE 2 DIABETES MODEL] Epoch 4/5 | Loss: 0.1359
[TYPE 2 DIABETES MODEL] Epoch 5/5 | Loss: 0.1350

===== TYPE 2 DIABETES MODEL RESULTS =====
Accuracy: 0.62884834663626
ROC AUC : 0.6369925796595373
F1 Score: 0.7551711169612636
Recall  : 0.6354430379746835


* Cleaned and encoded BRFSS survey data, handling missing values and converting categorical variables into numeric form
* Standardized features and split the dataset into training and testing sets
* Created two separate binary classification datasets:

  * Type 1 diabetes vs all other cases
  * Type 2 diabetes vs all other cases
* Trained two independent deep neural network models with dropout regularization
* Evaluated both models using accuracy, ROC-AUC, F1-score, and recall
